# 🔍 Para Pensar — Gradio y MediaPipe

**Materiales desarrollados por Matías Barreto, 2026**
**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura Oficial:** Procesamiento Digital de Imágenes
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

Este laboratorio responde las tres preguntas de reflexión del notebook `03_Integración_Gradio_y_MediaPipe`. Cada consigna tiene una respuesta conceptual seguida de código funcional que la demuestra en práctica.

| # | Tema |
|---|---|
| **1** | `gr.Interface` vs `gr.Blocks`: cuándo necesitamos cada uno |
| **2** | El concepto de Skill: ventajas de empaquetar conocimiento |
| **3** | Video en lugar de imagen: qué cambia en la integración |

> **Prerequisito:** tener ejecutado el Paso 1 del notebook `03_Integración_Gradio_y_MediaPipe` (instalación de dependencias). El modelo `face_landmarker.task` ya está disponible en esta carpeta.

## Microglosario

| Término | Definición |
|---|---|
| **`gr.Blocks`** | API de layout explícito de Gradio: permite múltiples funciones, botones y layouts personalizados en una sola interfaz |
| **Evento `.click()`** | Conexión entre un botón y una función Python; Gradio la ejecuta cuando el usuario presiona ese botón específico |
| **`RunningMode.VIDEO`** | Modo de FaceLandmarker que activa el tracker entre frames consecutivos para mayor estabilidad y velocidad |
| **`detect_for_video()`** | Método de FaceLandmarker para procesar un frame con su timestamp en ms; requiere que los frames lleguen en orden |
| **`cv2.VideoCapture`** | Clase de OpenCV que abre un archivo de video y permite leer sus frames uno a uno |
| **`cv2.VideoWriter`** | Clase de OpenCV que escribe frames procesados en un archivo de video de salida |
| **Skill** | Archivo `SKILL.md` que empaqueta conocimiento de dominio en formato estructurado, versionable con git y compartible entre agentes y equipos |

In [1]:
# Paso previo — importaciones y detector compartidos por las Consignas 1 y 3.
# Ejecutar esta celda antes de cualquiera de los bloques de consignas.
import os, urllib.request, tempfile
import gradio as gr
import mediapipe as mp
import cv2
import numpy as np

MODEL_PATH = "face_landmarker.task"
MODEL_URL  = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_landmarker/face_landmarker/float16/1/face_landmarker.task"
)

if not os.path.exists(MODEL_PATH):
    print("Descargando modelo face_landmarker.task ...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

tamanio_mb = os.path.getsize(MODEL_PATH) / 1_048_576
print(f"Modelo disponible: {tamanio_mb:.1f} MB")

# Aliases de la Tasks API — mismo patron que en el notebook de manos.
FaceLandmarker        = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
BaseOptions           = mp.tasks.BaseOptions
RunningMode           = mp.tasks.vision.RunningMode

# Detector en modo IMAGE, compartido por las Consignas 1 y 2.
# Se crea una vez para no recargar el modelo en cada llamada de la interfaz.
detector_imagen = FaceLandmarker.create_from_options(
    FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=RunningMode.IMAGE,
        num_faces=2,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5
    )
)

print("Entorno listo.")

c:\Proyectos\rodriguez-carmen-pdi-1c-2026\.venv_vision_aplicada\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modelo disponible: 3.6 MB
Entorno listo.


## Consigna 1 — `gr.Interface` vs `gr.Blocks`

### Respuesta

**No alcanza con `gr.Interface`. Necesitamos `gr.Blocks`.**

`gr.Interface` envuelve **una sola función** con entradas y salidas fijas. Agregar un segundo botón implica conectar dos funciones distintas al mismo componente de imagen de salida, algo que está fuera de su modelo.

```
gr.Interface  →  una función, layout automático
gr.Blocks     →  N funciones, layout explícito, eventos manuales
```

**Qué hay que cambiar:**

| Elemento | Paso 4 (`gr.Interface`) | Consigna 1 (`gr.Blocks`) |
|---|---|---|
| Constructor | `gr.Interface(fn=..., inputs=..., outputs=...)` | `with gr.Blocks() as demo:` |
| Componentes | declarados en `inputs=` / `outputs=` | instanciados dentro del bloque `with` |
| Funciones | una sola, fija | una por botón |
| Evento | automático al subir la imagen | `.click(fn=..., inputs=..., outputs=...)` por botón |

El componente de entrada (`img_entrada`) y el de salida (`img_salida`) son **compartidos** entre los dos botones. Gradio sabe cuál función llamar según el evento que se disparó.

In [2]:
# Consigna 1 — gr.Blocks con dos botones que comparten entrada y salida.

# ── Funcion A: landmarks ─────────────────────────────────────────────────

def detectar_landmarks(imagen_entrada):
    """Misma logica que en el Paso 4 del notebook fuente."""
    alto, ancho = imagen_entrada.shape[:2]
    imagen_mp   = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_entrada)
    resultado   = detector_imagen.detect(imagen_mp)
    imagen_out  = imagen_entrada.copy()
    for puntos in resultado.face_landmarks:
        for lm in puntos:
            cv2.circle(imagen_out, (int(lm.x * ancho), int(lm.y * alto)), 1, (0, 220, 180), -1)
    return imagen_out


# ── Funcion B: filtro de grises ──────────────────────────────────────────

def aplicar_grises(imagen_entrada):
    """Misma logica que en el Paso 2 del notebook fuente."""
    gris = cv2.cvtColor(imagen_entrada, cv2.COLOR_RGB2GRAY)
    # Gradio espera (H, W, 3); se replica el canal gris tres veces.
    return np.stack([gris, gris, gris], axis=-1)


# ── Interfaz con gr.Blocks ───────────────────────────────────────────────

with gr.Blocks(title="Landmarks vs Grises") as interfaz_c1:

    gr.Markdown("## Dos operaciones, una interfaz")
    gr.Markdown("Subi una fotografia y elegi que operacion aplicar.")

    with gr.Row():
        img_entrada = gr.Image(label="Imagen original", type="numpy")
        img_salida  = gr.Image(label="Resultado")
        # type="numpy": ambas funciones reciben y devuelven arrays NumPy.

    with gr.Row():
        btn_landmarks = gr.Button("Detectar landmarks", variant="primary")
        btn_grises    = gr.Button("Aplicar grises")
        # variant="primary" destaca visualmente el boton principal.

    # Cada boton queda conectado a su propia funcion.
    # `inputs` y `outputs` apuntan a los mismos componentes en ambos casos;
    # Gradio distingue la funcion a invocar por el evento que se disparo.
    btn_landmarks.click(fn=detectar_landmarks, inputs=img_entrada, outputs=img_salida)
    btn_grises.click(   fn=aplicar_grises,     inputs=img_entrada, outputs=img_salida)

interfaz_c1.launch(share=False)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


c:\Proyectos\rodriguez-carmen-pdi-1c-2026\.venv_vision_aplicada\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Proyectos\rodriguez-carmen-pdi-1c-2026\.venv_vision_aplicada\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Proyectos\rodriguez-carmen-pdi-1c-2026\.venv_vision_aplicada\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## Consigna 2 — El concepto de Skill

### Respuesta

Un **Skill** es un archivo `SKILL.md` que empaqueta conocimiento de dominio en formato estructurado, versionable y compartible. Comparado con escribir las instrucciones en el prompt de cada conversación nueva:

| Característica | Prompt en la conversación | Archivo `SKILL.md` |
|---|---|---|
| **Reutilización** | Hay que repetirlo en cada sesión | Se carga automáticamente cuando el agente lo necesita |
| **Historial** | No existe | `git log SKILL.md` muestra cada modificación |
| **Trabajo en equipo** | Cada persona tiene su copia, que diverge | Una sola fuente de verdad compartida |
| **Mantenimiento** | Cambiar algo implica actualizar en muchos lugares | Se edita en un lugar, se propaga a todos |
| **Contexto de ventana** | Ocupa tokens en cada conversación aunque no sea necesario | Se carga bajo demanda |

**Ventaja concreta más importante:** cuando el criterio de estilo de los materiales cambia (por ejemplo, se decide que el microglosario siempre va antes del código), basta con editar una línea en `SKILL.md`. Con el modelo de prompt, habría que avisar a cada integrante del equipo y esperar que actualice su copia.

> El notebook que estás leyendo *es un producto del Skill*: el estilo, la estructura y el nivel de comentarios siguen las instrucciones que el Skill define.

La celda de abajo busca el archivo `SKILL.md` en el repositorio y muestra su contenido, para ver en práctica cómo está estructurado el conocimiento empaquetado.

In [3]:
# Consigna 2 — Lectura del archivo SKILL.md.
# Este bloque no procesa imagenes; demuestra el concepto del Skill de forma interactiva.
import os

# El Skill puede estar en distintos niveles del repositorio; se prueba en orden.
posibles_rutas = [
    os.path.join(os.getcwd(), "SKILL.md"),
    os.path.join(os.path.dirname(os.getcwd()), "SKILL.md"),
    os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), "SKILL.md"),
]

ruta_skill = next((r for r in posibles_rutas if os.path.exists(r)), None)

if ruta_skill:
    print(f"SKILL.md encontrado: {ruta_skill}")
    print(f"Tamano: {os.path.getsize(ruta_skill):,} bytes\n")
    print("=" * 60)
    with open(ruta_skill, encoding="utf-8") as f:
        print(f.read())
    print("=" * 60)
else:
    # No encontrar el archivo no es un error: puede estar con otro nombre o en otra ruta.
    print("SKILL.md no encontrado en las rutas buscadas:")
    for r in posibles_rutas:
        print(f"  {r}")
    print()
    print("Esto no impide su uso: los agentes reciben la ruta del Skill")
    print("como parte de su configuracion, no buscan por nombre automaticamente.")

SKILL.md no encontrado en las rutas buscadas:
  c:\Proyectos\rodriguez-carmen-pdi-1c-2026\009 - vision_artificial_aplicada\003 - LAB\SKILL.md
  c:\Proyectos\rodriguez-carmen-pdi-1c-2026\009 - vision_artificial_aplicada\SKILL.md
  c:\Proyectos\rodriguez-carmen-pdi-1c-2026\SKILL.md

Esto no impide su uso: los agentes reciben la ruta del Skill
como parte de su configuracion, no buscan por nombre automaticamente.


## Consigna 3 — De imagen a video

### Respuesta

**Componente de Gradio:** `gr.Video` en lugar de `gr.Image`, tanto en entrada como en salida.

**Qué cambia en la función de procesamiento:**

| Elemento | Imagen estática | Video |
|---|---|---|
| **Entrada** | Array NumPy `(H, W, 3)` | Ruta de archivo `.mp4` (string) |
| **Salida** | Array NumPy procesado | Ruta de archivo de video procesado |
| **`RunningMode`** | `IMAGE` | `VIDEO` |
| **Método de detección** | `detector.detect(imagen_mp)` | `detector.detect_for_video(imagen_mp, timestamp_ms)` |
| **Timestamp** | No se usa | Milisegundo de cada frame: `frame_n × 1000 / fps` |
| **Lectura de frames** | No hace falta | `cv2.VideoCapture` frame a frame |
| **Escritura del resultado** | No hace falta | `cv2.VideoWriter` frame a frame |
| **Conversión de color** | No hace falta (Gradio entrega RGB) | BGR→RGB al leer, RGB→BGR al escribir |

**Por qué `RunningMode.VIDEO` y no `IMAGE`:**
En modo `VIDEO`, FaceLandmarker activa un *tracker* que conecta el rostro detectado en el frame anterior con el actual. Esto hace la detección más estable y más rápida (no re-corre el detector completo en cada frame). El precio es que los frames deben procesarse **en orden** y con timestamps **crecientes**.

```
Frame 0 (t=0 ms)   → detect_for_video() → deteccion completa (tracker inactivo)
Frame 1 (t=33 ms)  → detect_for_video() → tracker activo: busca cerca del bbox anterior
Frame 2 (t=66 ms)  → detect_for_video() → tracker activo
...
```

In [ ]:
# Consigna 3 — Detector de landmarks sobre video con gr.Video.

def procesar_video_landmarks(ruta_video_entrada):
    """
    Recibe la ruta de un archivo de video (entregada por gr.Video).
    Detecta landmarks en cada frame con FaceLandmarker en modo VIDEO.
    Devuelve la ruta del video procesado.
    """
    captura = cv2.VideoCapture(ruta_video_entrada)
    fps     = captura.get(cv2.CAP_PROP_FPS) or 30.0  # fallback si el codec no reporta fps
    ancho   = int(captura.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto    = int(captura.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Se escribe en un archivo temporal para no sobreescribir el video original.
    fd, ruta_salida = tempfile.mkstemp(suffix="_landmarks.mp4")
    os.close(fd)
    escritor = cv2.VideoWriter(
        ruta_salida, cv2.VideoWriter_fourcc(*"mp4v"), fps, (ancho, alto)
    )

    # El detector se crea dentro de la funcion para que el tracker se resetee
    # en cada video nuevo; con un detector compartido, el estado interno del tracker
    # del video anterior contaminaría la deteccion del siguiente.
    opciones_video = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=RunningMode.VIDEO,
        num_faces=2,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5
    )

    with FaceLandmarker.create_from_options(opciones_video) as detector:
        numero_frame = 0
        while captura.isOpened():
            ok, frame_bgr = captura.read()
            if not ok:
                break

            # VideoCapture entrega BGR; mp.Image necesita RGB.
            frame_rgb    = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            # El timestamp debe ser creciente y en ms; no puede repetirse entre frames.
            timestamp_ms = int(numero_frame * 1000 / fps)
            imagen_mp    = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            resultado    = detector.detect_for_video(imagen_mp, timestamp_ms)

            frame_anotado = frame_bgr.copy()
            for puntos_rostro in resultado.face_landmarks:
                for lm in puntos_rostro:
                    px = int(lm.x * ancho)
                    py = int(lm.y * alto)
                    cv2.circle(frame_anotado, (px, py), 1, (0, 220, 180), -1)
                    # Mismo color y tamano que en el Paso 4 y la Consigna 1.

            # VideoWriter escribe en BGR (formato nativo de OpenCV para video).
            escritor.write(frame_anotado)
            numero_frame += 1

    captura.release()
    escritor.release()
    return ruta_salida


interfaz_c3 = gr.Interface(
    fn=procesar_video_landmarks,
    inputs=gr.Video(label="Video de entrada"),
    # gr.Video entrega la ruta del archivo subido (string), no un array NumPy;
    # por eso la funcion usa VideoCapture en lugar de recibir el array directamente.
    outputs=gr.Video(label="Video con landmarks"),
    title="Detector de Landmarks en Video",
    description=(
        "Subi un video con uno o dos rostros. "
        "La aplicacion detecta los 478 landmarks en cada frame y devuelve el video procesado."
    ),
    flagging_mode="never"
)

interfaz_c3.launch(share=False)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


c:\Proyectos\rodriguez-carmen-pdi-1c-2026\.venv_vision_aplicada\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Proyectos\rodriguez-carmen-pdi-1c-2026\.venv_vision_aplicada\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
